이 자료는 위키독스 딥 러닝을 이용한 자연어 처리 입문의 GloVe 튜토리얼 자료입니다.  

링크 : https://wikidocs.net/22885  

이 자료는 2021년 10월 14일에 마지막으로 테스트되었습니다.

In [15]:
pip install glove_python_binary

ERROR: Could not find a version that satisfies the requirement glove_python_binary (from versions: none)
ERROR: No matching distribution found for glove_python_binary


In [2]:
pip list | grep glove

In [3]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
import urllib.request
import zipfile
from lxml import etree
import re
from nltk.tokenize import word_tokenize, sent_tokenize

In [5]:
# 데이터 다운로드
urllib.request.urlretrieve("https://raw.githubusercontent.com/GaoleMeng/RNN-and-FFNN-textClassification/master/ted_en-20160408.xml", filename="ted_en-20160408.xml")

('ted_en-20160408.xml', <http.client.HTTPMessage at 0x7965842ed400>)

In [7]:
targetXML = open('ted_en-20160408.xml', 'r', encoding='UTF8')
target_text = etree.parse(targetXML)

# xml 파일로부터 <content>와 </content> 사이의 내용만 가져온다.
parse_text = '\n'.join(target_text.xpath('//content/text()'))

# 정규 표현식의 sub 모듈을 통해 content 중간에 등장하는 (Audio), (Laughter) 등의 배경음 부분을 제거.
# 해당 코드는 괄호로 구성된 내용을 제거.
content_text = re.sub(r'\([^)]*\)', '', parse_text)

# NLTK의 sent_tokenize에서 필요한 'punkt_tab' 리소스를 다운로드합니다.
import nltk
nltk.download('punkt_tab', quiet=True)

# 입력 코퍼스에 대해서 NLTK를 이용하여 문장 토큰화를 수행.
sent_text = sent_tokenize(content_text)

# 각 문장에 대해서 구두점을 제거하고, 대문자를 소문자로 변환.
normalized_text = []
for string in sent_text:
     tokens = re.sub(r"[^a-z0-9]+", " ", string.lower())
     normalized_text.append(tokens)

# 각 문장에 대해서 NLTK를 이용하여 단어 토큰화를 수행.
result = [word_tokenize(sentence) for sentence in normalized_text]

In [8]:
print('총 샘플의 개수 : {}'.format(len(result)))

총 샘플의 개수 : 273424


In [14]:
!pip install gensim

from gensim.models import Word2Vec

# Word2Vec 모델 훈련 (GloVe와 유사한 목적으로 사용)
# vector_size는 GloVe의 no_components에 해당, window는 동시 등장 윈도우,
# min_count는 학습에 포함할 단어의 최소 빈도, workers는 쓰레드 수, epochs는 학습 에포크 수
model = Word2Vec(sentences=result, vector_size=100, window=5, min_count=1, workers=4, epochs=20)

# (원래 GloVe에서 사용했던 dictionary 추가는 Word2Vec에서는 필요 없음)
# (gensim.models.Word2Vec은 자체적으로 단어와 벡터를 관리)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.6 MB/s eta 0:00:00


In [17]:
print(model.wv.most_similar("man"))

[('woman', 0.7680018544197083), ('guy', 0.7177895307540894), ('soldier', 0.6854215264320374), ('boy', 0.6463778018951416), ('gentleman', 0.6412402987480164), ('lady', 0.6320480108261108), ('girl', 0.6296144723892212), ('person', 0.6198557019233704), ('rabbi', 0.6096575260162354), ('pianist', 0.6014041304588318)]


In [18]:
print(model.wv.most_similar("boy"))

[('girl', 0.8682534098625183), ('kid', 0.7950125932693481), ('woman', 0.7122066617012024), ('lady', 0.669894278049469), ('man', 0.6463778018951416), ('sister', 0.6244258284568787), ('guy', 0.6137129068374634), ('dancer', 0.5944465398788452), ('daughter', 0.5865625739097595), ('princess', 0.5855908989906311)]


In [19]:
print(model.wv.most_similar("university"))

[('harvard', 0.8456655740737915), ('stanford', 0.8198503255844116), ('ucla', 0.7986131906509399), ('cambridge', 0.7895934581756592), ('princeton', 0.7708893418312073), ('mit', 0.7636074423789978), ('cornell', 0.7614110112190247), ('berkeley', 0.7599048018455505), ('mellon', 0.7570579051971436), ('nyu', 0.7505486011505127)]


In [20]:
print(model.wv.most_similar("water"))

[('air', 0.71988844871521), ('soil', 0.7197558879852295), ('heat', 0.7067183256149292), ('moisture', 0.6910398602485657), ('oxygen', 0.6859487891197205), ('nutrients', 0.6710667014122009), ('sunlight', 0.6416147351264954), ('liquid', 0.6414562463760376), ('seawater', 0.63723224401474), ('nitrogen', 0.6292104721069336)]


In [21]:
print(model.wv.most_similar("physics"))

[('biology', 0.7207955121994019), ('theoretical', 0.7148163318634033), ('mechanics', 0.6816085577011108), ('mathematics', 0.6767722368240356), ('quantum', 0.6725067496299744), ('neuroscience', 0.6667296886444092), ('genetics', 0.6300806403160095), ('economics', 0.621124804019928), ('theory', 0.6178725361824036), ('science', 0.6168698668479919)]


In [22]:
print(model.wv.most_similar("muscle"))

[('skeletal', 0.7529048919677734), ('tissue', 0.7304844856262207), ('nerve', 0.7020781636238098), ('metastases', 0.6845554709434509), ('liver', 0.6601989269256592), ('tumor', 0.6457127332687378), ('dopamine', 0.6131149530410767), ('angiogenesis', 0.6121384501457214), ('limb', 0.611285388469696), ('muscles', 0.6067396998405457)]


In [23]:
print(model.wv.most_similar("clean"))

[('supply', 0.6103049516677856), ('heating', 0.6012115478515625), ('supplies', 0.573658287525177), ('burn', 0.5511466264724731), ('sewage', 0.5495938658714294), ('electricity', 0.5450708270072937), ('contaminated', 0.542003870010376), ('sanitation', 0.5417504906654358), ('fresh', 0.5413352251052856), ('renewable', 0.5393579006195068)]
